# FAIR 2026: chạy repo trên Google Colab

Notebook này là **cách khởi động tùy chọn**. Code chính nằm trong `src/fair2026_lms_kan/` và cũng chạy trên máy/server có PyTorch. Chạy lần lượt từng cell. Colab cần quyền truy cập Drive chứa manifest, ảnh và checkpoint của bạn.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Lấy repo

Điền URL GitHub sau khi đã tạo repo. Nếu chưa đăng GitHub, để trống và upload gói ZIP đã được chuẩn bị.

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile
REPO_URL = ''  # Ví dụ: 'https://github.com/username/repository.git'
if REPO_URL:
    repo_name = REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
    repo = Path('/content') / repo_name
    if not repo.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    archive = next((name for name in uploaded if name.lower().endswith('.zip')), None)
    if not archive:
        raise ValueError('Hãy upload ZIP của repo hoặc điền REPO_URL.')
    with zipfile.ZipFile(archive) as z:
        if any(Path(item).is_absolute() or '..' in Path(item).parts for item in z.namelist()):
            raise ValueError('ZIP chứa đường dẫn không an toàn.')
        z.extractall('/content')
    repo = Path('/content/fair2026_public_code')
if not (repo / 'pyproject.toml').exists():
    raise FileNotFoundError(f'Không thấy pyproject.toml trong {repo}')
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Repo:', repo)

## 2. Gắn Drive và khai báo đường dẫn

Sửa `DRIVE_ROOT` nếu tài khoản Drive đặt thư mục `colab` ở nơi khác. Không đưa file `configs/local.yaml` lên GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import yaml
DRIVE_ROOT = Path('/content/drive/MyDrive/colab')
config = yaml.safe_load(Path('configs/default.yaml').read_text())
config['paths'].update({
    'manifest': str(DRIVE_ROOT / 'LMS_KAN_pipeline/checkpoints/fair_2026_evidence/rsna_split_manifest_used.csv'),
    'rsna_raw_root': str(DRIVE_ROOT / 'Kaggle_Datasets/dataset/rsna_raw'),
    'lung_masks_root': str(DRIVE_ROOT / 'Kaggle_Datasets/dataset/lung_masks'),
    'rsna_root': str(DRIVE_ROOT / 'Kaggle_Datasets/dataset/rsna_pool'),
    'kermany_root': str(DRIVE_ROOT / 'Kaggle_Datasets/dataset/kermany'),
    'segmentation_checkpoint': str(DRIVE_ROOT / 'LMS_KAN_pipeline/checkpoints/mobileunet_final.pt'),
    'crop_cache': str(DRIVE_ROOT / 'LMS_KAN_pipeline/cache/fair_rsna_lung_crop_cache'),
    'output_dir': str(DRIVE_ROOT / 'LMS_KAN_pipeline/github_runs'),
})
for key in ('manifest', 'rsna_root', 'kermany_root', 'segmentation_checkpoint'):
    path = Path(config['paths'][key])
    print(key, 'OK' if path.exists() else 'MISSING', path)
    # Pipeline sẽ kiểm tra đầy đủ; nếu chưa có dataset, bật DOWNLOAD_MISSING.
Path('configs/local.yaml').write_text(yaml.safe_dump(config), encoding='utf-8')

## 3. Kiểm tra code và khả năng nạp checkpoint

Đây là kiểm tra quan trọng trước khi train. Nó thử ba head, MobileUNet và nạp nghiêm ngặt checkpoint FastKAN cũ. Nếu báo lỗi, dừng tại đây và gửi traceback để sửa; không chạy thí nghiệm dài.

In [ ]:
primary_checkpoint = DRIVE_ROOT / 'LMS_KAN_pipeline/checkpoints/multiseed/seed_42.pt'
cmd = [sys.executable, 'tests/smoke_runtime.py']
if primary_checkpoint.exists():
    cmd += ['--head', 'fastkan', '--classifier-checkpoint', str(primary_checkpoint)]
if Path(config['paths']['segmentation_checkpoint']).exists():
    cmd += ['--segmentation-checkpoint', config['paths']['segmentation_checkpoint']]
subprocess.run(cmd, check=True)

## 4. Pipeline thí nghiệm

Chạy thử `--dry-run` để xem thứ tự. `RUN_PIPELINE=True` sẽ chạy 12 job (3 head × 3 seed và 3 FastKAN không crop), bỏ qua job đã hoàn tất, tiếp tục job có `last.pt`. `MAX_JOBS=3` giới hạn ba job chưa xong mỗi phiên; chạy lại cùng cell ở tài khoản/phiên Colab tiếp theo. Nếu dữ liệu chưa có, đặt `DOWNLOAD_MISSING=True`; cần đăng nhập Kaggle và chấp nhận điều khoản RSNA trước. Nếu không còn manifest gốc, `REBUILD_RSNA=True` sẽ tạo split **mới**, không thể xem là split của bài đã nộp. Giữ Drive gắn và kiểm tra đường dẫn trước khi chạy.


In [ ]:
RUN_PIPELINE = False
DOWNLOAD_MISSING = False  # Bật khi thiếu dữ liệu Kaggle.
REBUILD_RSNA = False      # Chỉ bật nếu chấp nhận tạo split mới từ DICOM.
MAX_JOBS = 3
MAX_HOURS = 5.5

plan = [sys.executable, 'pipeline.py', '--config', 'configs/local.yaml', '--dry-run']
subprocess.run(plan, check=True)
if RUN_PIPELINE:
    cmd = [sys.executable, 'pipeline.py', '--config', 'configs/local.yaml',
           '--max-jobs', str(MAX_JOBS), '--max-hours', str(MAX_HOURS)]
    if DOWNLOAD_MISSING:
        cmd.append('--download-missing')
    if REBUILD_RSNA:
        cmd.append('--rebuild-rsna')
    subprocess.run(cmd, check=True)
else:
    print('Chưa chạy. Đổi RUN_PIPELINE=True sau khi kiểm tra plan và đường dẫn.')


## 5. Chỉ đánh giá checkpoint hiện có (tùy chọn)

Chỉ dùng checkpoint do nhóm bạn tạo và tin cậy. Đánh giá trên RSNA test và toàn bộ Kermany có thể tốn thời gian, nhất là khi tạo crop cache lần đầu.

In [ ]:
RUN_EVALUATE = False
CHECKPOINT_PATH = primary_checkpoint
if RUN_EVALUATE:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(CHECKPOINT_PATH)
    subprocess.run([sys.executable, 'evaluate.py',
                    str(CHECKPOINT_PATH), '--config', 'configs/local.yaml',
                    '--output-dir', str(DRIVE_ROOT / 'LMS_KAN_pipeline/github_evaluation')], check=True)
else:
    print('Chưa đánh giá. Đổi RUN_EVALUATE=True khi cần.')